In [2]:
import numpy as np

# -------------------------------
# Step 1: Create a Custom Deterministic Cliff Walking Environment
# -------------------------------
class CliffWalkingDeterministic:
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.observation_space_n = self.rows * self.cols  # 48 states
        self.action_space_n = 4

        # Define specific key states
        self.start_state = 36  # Bottom-left: (3, 0)
        self.goal_state = 47   # Bottom-right: (3, 11)
        self.cliff_states = set(range(37, 47))  # Bottom row except start and goal

        # Actions: 0: Left, 1: Down, 2: Right, 3: Up
        self.actions = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}
        self.P = {s: {a: [] for a in range(self.action_space_n)} for s in range(self.observation_space_n)}
        self._build_transition_model()

    def _build_transition_model(self):
        for state in range(self.observation_space_n):
            # Goal state is terminal
            if state == self.goal_state:
                for action in range(self.action_space_n):
                    self.P[state][action] = [(1.0, state, 0.0, True)]
                continue

            row, col = state // self.cols, state % self.cols

            for action, (dr, dc) in self.actions.items():
                new_row = max(0, min(row + dr, self.rows - 1))
                new_col = max(0, min(col + dc, self.cols - 1))
                next_state = new_row * self.cols + new_col

                # Check if the next state falls into the Cliff
                if next_state in self.cliff_states:
                    # Going into the cliff gives a massive penalty and teleports back to start
                    reward = -100.0
                    done = False
                    self.P[state][action] = [(1.0, self.start_state, reward, done)]
                else:
                    # Every safe step costs -1 to encourage finding the shortest path
                    reward = -1.0
                    done = (next_state == self.goal_state)
                    self.P[state][action] = [(1.0, next_state, reward, done)]

# Instantiate the new environment
env = CliffWalkingDeterministic()
print(f"Number of states: {env.observation_space_n} (4x12 Grid)")
print(f"Number of actions: {env.action_space_n}")

# -------------------------------
# Step 2: Policy Evaluation
# -------------------------------
def policy_evaluation(policy, value_table, env, gamma=0.9, theta=1e-10):
    while True:
        delta = 0
        for state in range(env.observation_space_n):
            v = 0
            action = policy[state]
            for prob, next_state, reward, done in env.P[state][action]:
                v += prob * (reward + gamma * value_table[next_state])

            delta = max(delta, abs(value_table[state] - v))
            value_table[state] = v

        if delta < theta:
            break
    return value_table

# -------------------------------
# Step 3: Policy Improvement
# -------------------------------
def policy_improvement(value_table, policy, env, gamma=0.9):
    policy_stable = True
    for state in range(env.observation_space_n):
        old_action = policy[state]
        action_values = np.zeros(env.action_space_n)

        for action in range(env.action_space_n):
            for prob, next_state, reward, done in env.P[state][action]:
                action_values[action] += prob * (reward + gamma * value_table[next_state])

        policy[state] = np.argmax(action_values)
        if old_action != policy[state]:
            policy_stable = False

    return policy, policy_stable

# -------------------------------
# Step 4: Policy Iteration Execution Loop
# -------------------------------
def policy_iteration(env, gamma=0.9, theta=1e-10):
    policy = np.zeros(env.observation_space_n, dtype=int)
    value_table = np.zeros(env.observation_space_n)

    while True:
        value_table = policy_evaluation(policy, value_table, env, gamma, theta)
        policy, policy_stable = policy_improvement(value_table, policy, env, gamma)
        if policy_stable:
            return policy, value_table

# Run Policy Iteration on Cliff Walking
optimal_policy, optimal_value = policy_iteration(env)

# Readable Action Mapping
action_names = {0: "←", 1: "↓", 2: "→", 3: "↑"}
policy_grid = np.array([action_names[a] for a in optimal_policy]).reshape(env.rows, env.cols)

# Format the grid layout for easier structural visualization
policy_grid[3, 0] = '🏁'  # Start State marker
for c in range(1, 11):
    policy_grid[3, c] = '💀' # Cliff markers
policy_grid[3, 11] = '🏆' # Goal State marker

print("\nOptimal Policy Direction Grid (💀 = Cliff, 🏁 = Start, 🏆 = Goal):")
print(policy_grid)

print("\nOptimal Value Function Matrix (rounded):")
print(np.round(optimal_value, 2).reshape(env.rows, env.cols))


Number of states: 48 (4x12 Grid)
Number of actions: 4

Optimal Policy Direction Grid (💀 = Cliff, 🏁 = Start, 🏆 = Goal):
[['↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓']
 ['↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓' '↓']
 ['→' '→' '→' '→' '→' '→' '→' '→' '→' '→' '→' '↓']
 ['🏁' '💀' '💀' '💀' '💀' '💀' '💀' '💀' '💀' '💀' '💀' '🏆']]

Optimal Value Function Matrix (rounded):
[[-7.71 -7.46 -7.18 -6.86 -6.51 -6.13 -5.7  -5.22 -4.69 -4.1  -3.44 -2.71]
 [-7.46 -7.18 -6.86 -6.51 -6.13 -5.7  -5.22 -4.69 -4.1  -3.44 -2.71 -1.9 ]
 [-7.18 -6.86 -6.51 -6.13 -5.7  -5.22 -4.69 -4.1  -3.44 -2.71 -1.9  -1.  ]
 [-7.46 -7.18 -6.86 -6.51 -6.13 -5.7  -5.22 -4.69 -4.1  -3.44 -1.    0.  ]]
